# Import FRED data
- Author: Bryan Bravo
- Created: 2026-03-02
## Import Libraries

In [1]:
import os
import sys

# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


### Custom Functions

In [41]:
def fetch_fred_data(
    api_key: str,
    currency_name: str, # Currency name of series id.
    series_id: str,  # FRED series ID
    rate_name: str = None,
    start_date: str = "2002-04-01",
    end_date: str = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")
    ) -> pd.DataFrame:
    """
    Fetches daily foreign exchange rate data from the FRED API, cleans it, and returns
    a standardized DataFrame containing the latest revision for each date.

    This function:
    - Adds a `currency` column for identification
    - Computes a unified USD exchange rate (`us_fx_rate`) so that:
        • If the series is FX-per-USD (e.g., DEXJPUS), it inverts the value
        • Otherwise, it uses the value as-is

    Parameters
    ----------
    api_key : str
        FRED API key used for authentication.
    currency_name : str
        Human-readable currency name to attach to the output (e.g., "euro", "yen").
    series_id : str
        FRED series ID (e.g., "DEXUSEU", "DEXJPUS").
    rate_name : str
        If API value is based on interest rates, this will create a column name matching the string value.
    start_date : str
        Start date for the query in YYYY-MM-DD format. (Defaults to "2002-01-01")
    end_date : str
        End date for the query in YYYY-MM-DD format. (Defaults to the last day of the previous month)

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns:
        - date : datetime64
        - currency : str
        - `rate_name` <- value derived from variable given : float
        representing the latest available FX rate for each date.

    Raises
    ------
    Exception
        If the API request fails or the response cannot be parsed.

    """

    try:
        response = requests.get(
            f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&realtime_start={start_date}&realtime_end={end_date}&api_key={api_key}&file_type=json",
            timeout=10)
        fred_data = response.json()

        if 'error_code' in fred_data:  # check if response is None
            print(f"✗ FRED API error: {fred_data.get('error_code', 'error_message')}")

        # Create pandas DataFrame from observations
        df = pd.DataFrame(fred_data['observations'])

        # Select relevant columns
        df = df[['date', 'realtime_start', 'value']]
        df['currency'] = currency_name


        # Correct data types and add joining variables
        df['date'] = pd.to_datetime(df['date'])
        df['realtime_start'] = pd.to_datetime(df['realtime_start'])
        df['value'] = pd.to_numeric(df['value'], errors='coerce')  # Convert to numeric, coerce errors to NaN

        df['join_dt'] = df['date'].dt.strftime("%Y%m").astype(int)
        # Keep only the most recent revision for each date
        df = df.loc[df.groupby('date')['realtime_start'].idxmax()]
        df = df[df['date'] >= start_date]

        # Add USD exchange rate (USD/FX)
        if rate_name == 'fx_rate':
            if series_id[5:] == "US":
                df['fx_rate'] = 1 / df['value']
            else:
                df['fx_rate'] = df['value']

            # subset columns
            df = df[['date', 'currency', 'fx_rate', 'join_dt']].reset_index(drop=True)

        # Add column with identified rate name
        else:
            df[rate_name] = df['value']
            # subset columns
            df = df[['date', 'currency', rate_name, 'join_dt']].reset_index(drop=True)
        

        print(f"""
    ✓ Fetched {len(df)} records for '{currency_name}': {rate_name}
    from FRED for years {dt.strftime(df['date'].min(), '%Y-%m-%d')} through {dt.strftime(df['date'].max(), '%Y-%m-%d')}
    ******************************************************************************""")
        return df
    except Exception as e:
        print(f"✗ Error fetching FRED data on {currency_name}: {str(e)[:100]}")

### Variables

In [42]:
fred_api = hardcoded_keys.FRED_API_KEY

## Query

In [59]:
fred_fx_series = {
    "euro": {
        'interest_rate_24h': 'IRSTCI01EZM156N',
        'interest_rate_90d': 'IR3TIB01EZM156N',
        'interest_rate_10y': 'IRLTLT01EZM156N',
        'fx_rate': 'DEXUSEU'
    },
    "japan": {
        'interest_rate_24h': 'IRSTCI01JPM156N',
        'interest_rate_90d': 'IR3TIB01JPM156N',
        'interest_rate_10y': 'IRLTLT01JPM156N',
        'fx_rate': 'DEXJPUS'
    },
    # "china": "DEXCHUS",
    # "canada": "DEXCAUS",
    # "united_kingdom": "DEXUSUK",
    # "south_korea": "DEXKOUS",
    # "mexico": "DEXMXUS",
    # "india": "DEXINUS",
    # "brazil": "DEXBZUS",
    # "australia": "DEXUSAL",
    # "switzerland": "DEXSZUS",
    # "sweden": "DEXSDUS",
    # "norway": "DEXNOUS",
    # "denmark": "DEXDNUS",
    # "hong_kong": "DEXHKUS",
    # "singapore": "DEXSIUS",
    # "south_africa": "DEXSFUS",
    # "taiwan": "DEXTAUS",
    # "thailand": "DEXTHUS",
    # "new_zealand": "DEXUSNZ",
    # "malaysia": "DEXMAUS",
    # "venezuela": "DEXVZUS",
    # "united_states": "DTWEXBGS"  # Nominal Broud U.S. Dollar Index
}

fred_df = pd.DataFrame([[]])
for country, kval in fred_fx_series.items():
    

    for i, (rate_name, series) in enumerate(kval.items()):
        if i == 0:
            df1 = fetch_fred_data(fred_api, country, series, rate_name)
        else:
            df2 = fetch_fred_data(fred_api, country, series, rate_name)
            df1 = (df1.drop('date', axis=1)
                   .merge(df2, how='outer', on=['join_dt', 'currency']))

        fred_df = pd.concat([fred_df, df1], ignore_index=True)

# Convert to PySpark df
fred_df = (
    spark.createDataFrame(fred_df.drop('join_dt', axis=1))
    .withColumn('date', F.date_format(F.col('date'), "yyyyMMdd").cast('int'))
    .dropna())
fred_df.cache().count()


    ✓ Fetched 286 records for 'euro': interest_rate_24h
    from FRED for years 2002-04-01 through 2026-01-01
    ******************************************************************************

    ✓ Fetched 286 records for 'euro': interest_rate_90d
    from FRED for years 2002-04-01 through 2026-01-01
    ******************************************************************************

    ✓ Fetched 286 records for 'euro': interest_rate_10y
    from FRED for years 2002-04-01 through 2026-01-01
    ******************************************************************************

    ✓ Fetched 6235 records for 'euro': fx_rate
    from FRED for years 2002-04-01 through 2026-02-20
    ******************************************************************************

    ✓ Fetched 286 records for 'japan': interest_rate_24h
    from FRED for years 2002-04-01 through 2026-01-01
    ******************************************************************************

    ✓ Fetched 286 records for 'japan'

11950